# Perfilado inicial — SECOP II Contratos Electrónicos (`jbjy-vk9h`)

Objetivo: responder las ocho preguntas de perfilado de `CLAUDE.md`, que son el
insumo de los ADR de la Fase 1 (incrementalidad, particionado, formato) y de la
estrategia de deduplicación.

Estrategia: lo que se puede contar exacto se cuenta **en el servidor** con SoQL
(volumen, nulos, duplicados). Lo que necesita mirar valores (formatos, nulos
disfrazados, fechas) se hace sobre una **muestra de 50.000 filas**.



In [1]:
import sys

print(sys.executable)
print(sys.version)

c:\Users\Usuario\proyectos\secop-data-platform\.venv\Scripts\python.exe
3.12.10 (tags/v3.12.10:0cc8128, Apr  8 2025, 12:21:36) [MSC v.1943 64 bit (AMD64)]


In [2]:
import json
import os
import re
import unicodedata
from pathlib import Path

import pandas as pd
import requests
from dotenv import load_dotenv

load_dotenv()

DATASET_ID = "jbjy-vk9h"
BASE_URL = f"https://www.datos.gov.co/resource/{DATASET_ID}.json"
METADATA_URL = f"https://www.datos.gov.co/api/views/{DATASET_ID}.json"
APP_TOKEN = os.getenv("SOCRATA_APP_TOKEN")

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

pd.set_option("display.max_rows", 120)
pd.set_option("display.max_colwidth", 80)

In [3]:
def soql(params):
    """Ejecuta una consulta SoQL y devuelve el resultado como DataFrame."""
    headers = {"X-App-Token": APP_TOKEN} if APP_TOKEN else {}
    response = requests.get(BASE_URL, params=params, headers=headers, timeout=300)
    response.raise_for_status()
    return pd.DataFrame(response.json())

## 1. Volumen total y número de columnas

In [4]:
metadata = requests.get(METADATA_URL, timeout=60).json()
columns = pd.DataFrame(metadata["columns"])[["fieldName", "dataTypeName"]]

total_rows = int(soql({"$select": "count(*)"}).iloc[0, 0])
rows_updated_at = pd.to_datetime(metadata["rowsUpdatedAt"], unit="s", utc=True)

print(f"Filas: {total_rows:,}")
print(f"Columnas: {len(columns)}")
print(f"Última actualización del dataset (metadatos): {rows_updated_at}")
columns["dataTypeName"].value_counts()

Filas: 5,965,816
Columnas: 95
Última actualización del dataset (metadatos): 2026-10-08 18:58:29+00:00


dataTypeName
text             64
number           19
calendar_date    11
url               1
Name: count, dtype: int64

## 2. Columna candidata a marca de agua

Una marca de agua sirve para pedirle a la fuente "solo lo que cambió desde la
última carga". Para que sea confiable, la columna tiene que cumplir cuatro
criterios:

1. **Completa.** Si está nula, esa fila nunca entra en una carga incremental.
2. **Cambia solo cuando la fila cambia.** Si no, las modificaciones retroactivas se pierden.
3. **Granularidad suficiente.** Con solo fecha, una carga que corre a mitad del
   día no sabe qué parte de ese día ya leyó.
4. **Coherente.** Una "última actualización" no puede ser anterior a la firma del contrato.

Candidatas: los campos de sistema de Socrata (`:created_at`, `:updated_at`) y la
columna de negocio `ultima_actualizacion`.

### 2.1 Campos de sistema de Socrata

Registran cuándo se escribió la fila **en el portal**, no cuándo cambió el
contrato. Si el mínimo y el máximo coinciden, todas las filas se escribieron en
el mismo instante.

In [5]:
soql({
    "$select": "min(:created_at) as min_creado, max(:created_at) as max_creado, "
               "min(:updated_at) as min_actualizado, max(:updated_at) as max_actualizado"
})

,min_creado,max_creado,min_actualizado,max_actualizado
0,2026-10-08T16:47:32.753Z,2026-10-08T16:47:32.753Z,2026-10-08T16:47:32.753Z,2026-10-08T16:47:32.753Z


**Lectura.** El 2026-10-08 los cuatro valores eran idénticos: el portal
reemplaza el dataset completo en cada carga. `:updated_at` dice cuándo se recargó
el portal, no cuándo cambió cada contrato, así que falla el criterio 2.
**Descartada.**

### 2.2 `ultima_actualizacion` — criterio 1: completitud

In [6]:
watermark = soql({
    "$select": "count(ultima_actualizacion) as no_nulos, "
               "min(ultima_actualizacion) as minimo, max(ultima_actualizacion) as maximo"
})
non_null = int(watermark.loc[0, "no_nulos"])
print(f"ultima_actualizacion no nula: {non_null:,} de {total_rows:,} ({non_null / total_rows:.1%})")
watermark

ultima_actualizacion no nula: 3,477,546 de 5,965,816 (58.3%)


,no_nulos,minimo,maximo
0,3477546,2016-01-29T00:00:00.000,2026-10-06T00:00:00.000


¿Los nulos se concentran en algún estado del contrato? Si fueran solo
borradores, podrían ignorarse; si incluyen contratos en ejecución, no.

In [7]:
by_status = soql({
    "$select": "estado_contrato, count(*) as total",
    "$group": "estado_contrato",
})
nulls_by_status = soql({
    "$select": "estado_contrato, count(*) as nulos",
    "$where": "ultima_actualizacion IS NULL",
    "$group": "estado_contrato",
})

by_status = by_status.merge(nulls_by_status, on="estado_contrato", how="left").fillna({"nulos": "0"})
by_status[["total", "nulos"]] = by_status[["total", "nulos"]].astype(int)
by_status["pct_nulos"] = (by_status["nulos"] / by_status["total"] * 100).round(1)
by_status.sort_values("total", ascending=False)

,estado_contrato,total,nulos,pct_nulos
6,En ejecución,1732613,1723636,99.5
4,Cerrado,1709266,66862,3.9
8,Modificado,1086758,41430,3.8
11,terminado,779089,33474,4.3
1,Borrador,243416,243416,100.0
0,Aprobado,203901,202399,99.3
2,Cancelado,109012,108971,100.0
7,enviado Proveedor,43496,43496,100.0
3,cedido,27567,817,3.0
5,En aprobación,23631,23631,100.0


In [8]:
# Distribución por año: ¿los nulos son contratos viejos, anteriores a que existiera la columna?
soql({
    "$select": "date_extract_y(ultima_actualizacion) as anio, count(*) as filas",
    "$group": "anio",
    "$order": "anio",
})

,anio,filas
0,2016,131
1,2017,1497
2,2018,31549
3,2019,59572
4,2020,85917
5,2021,160609
6,2022,276042
7,2023,665905
8,2024,649227
9,2025,705012


### 2.3 `ultima_actualizacion` — criterio 3: granularidad

Si ninguna fila tiene hora distinta de 00:00, la columna solo guarda la fecha.

In [9]:
soql({
    "$select": "count(*) as filas_con_hora",
    "$where": "date_extract_hh(ultima_actualizacion) != 0 OR date_extract_mm(ultima_actualizacion) != 0",
})

,filas_con_hora
0,0


### 2.4 `ultima_actualizacion` — criterio 4: coherencia

In [10]:
comparable = int(soql({
    "$select": "count(*)",
    "$where": "ultima_actualizacion IS NOT NULL AND fecha_de_firma IS NOT NULL",
}).iloc[0, 0])
updated_before_signed = int(soql({
    "$select": "count(*)",
    "$where": "ultima_actualizacion < fecha_de_firma",
}).iloc[0, 0])
print(f"Actualizadas antes de firmarse: {updated_before_signed:,} de {comparable:,} comparables")

Actualizadas antes de firmarse: 6 de 3,477,546 comparables


### 2.5 Criterio 2 — pendiente

No se puede verificar con una sola foto del dataset. Hace falta descargar el
mismo conjunto de contratos en dos días distintos y comprobar si, cuando cambia
algún campo, también cambia `ultima_actualizacion`.

### Conclusión de la sección 2

Ninguna candidata cumple los cuatro criterios (datos del 2026-10-09):

* `:updated_at` falla el criterio 2: refleja la recarga del portal, no el cambio del contrato.
* `ultima_actualizacion` falla el criterio 1, y de la peor forma: está nula en el
  ≈42 % de las filas, pero esos nulos no están repartidos. Se concentran en los
  contratos en ejecución (99,5 % nulos), borradores, aprobados y cancelados,
  mientras que en cerrados, modificados y terminados ronda el 4 %. Es decir,
  falta justo en los contratos que todavía van a cambiar. Hipótesis **no
  verificada**: el portal solo la llena cuando el contrato se modifica o se cierra.
* También falla el criterio 3: solo guarda fecha, sin hora.
* El criterio 4 se cumple casi siempre: 6 inconsistencias entre 3,48 millones.
* El criterio 2 está sin verificar.

Consecuencia para el ADR de incrementalidad: "pedir solo lo posterior a la marca
de agua" no alcanza por sí solo. Las alternativas a evaluar en el ADR —no aquí—
incluyen extraer completo y comparar contra la carga anterior, o usar
`ultima_actualizacion` con una ventana de reproceso más un mecanismo aparte para
las filas sin fecha.

## 3. Clave de negocio única

Una clave de negocio tiene que cumplir tres criterios:

1. **Nunca nula.**
2. **Única:** una fila por contrato.
3. **Estable:** el mismo contrato conserva la misma clave entre cargas.

Candidatas: `id_contrato`, `referencia_del_contrato` y `proceso_de_compra`.

### 3.1 Criterios 1 y 2 para las tres candidatas

`count(col)` cuenta los no nulos y `count(distinct col)` los valores distintos.
Si son iguales, la columna es única entre las filas que la tienen.

In [11]:
key_candidates = ["id_contrato", "referencia_del_contrato", "proceso_de_compra"]

rows = []
for col in key_candidates:
    result = soql({"$select": f"count({col}) as no_nulos, count(distinct {col}) as distintos"}).iloc[0].astype(int)
    rows.append({"columna": col, "no_nulos": result["no_nulos"], "distintos": result["distintos"]})

keys = pd.DataFrame(rows)
keys["pct_nulos"] = ((1 - keys["no_nulos"] / total_rows) * 100).round(2)
keys["repetidos"] = keys["no_nulos"] - keys["distintos"]
keys

,columna,no_nulos,distintos,pct_nulos,repetidos
0,id_contrato,5965816,5965510,0.0,306
1,referencia_del_contrato,5965816,4401372,0.0,1564444
2,proceso_de_compra,5965816,5254029,0.0,711787


### 3.2 Los repetidos de `id_contrato`

In [12]:
duplicated_ids = soql({
    "$select": "id_contrato, count(*) as veces",
    "$group": "id_contrato",
    "$having": "count(*) > 1",
    "$order": "veces DESC",
    "$limit": 50000,
})
print(f"id_contrato con más de una fila: {len(duplicated_ids)}")
duplicated_ids["veces"].value_counts()

id_contrato con más de una fila: 306


veces
2    306
Name: count, dtype: int64

¿Son copias exactas o versiones distintas del mismo contrato? Si son copias,
deduplicar no pierde información. Si son versiones, hay que decidir cuál gana,
y eso conecta con la marca de agua.

Se revisan **todos** los repetidos, en lotes de 50 para no armar una URL gigante.

In [13]:
ids = duplicated_ids["id_contrato"].tolist()
id_batches = [ids[i:i + 50] for i in range(0, len(ids), 50)]

parts = []
for batch in id_batches:
    id_list = ", ".join(f"'{i}'" for i in batch)
    parts.append(soql({"$where": f"id_contrato IN ({id_list})", "$limit": 1000}))
duplicated_rows = pd.concat(parts, ignore_index=True)

# astype(str): algunas columnas llegan como objetos JSON (dict) y pandas no puede compararlos
after_dedup = duplicated_rows.astype(str).drop_duplicates()["id_contrato"].value_counts()
not_identical = after_dedup[after_dedup > 1]

print(f"Filas traídas: {len(duplicated_rows)} (esperadas: {duplicated_ids['veces'].astype(int).sum()})")
print(f"id_contrato con filas que difieren: {len(not_identical)} de {len(ids)}")

Filas traídas: 612 (esperadas: 612)
id_contrato con filas que difieren: 0 de 306


### 3.3 Criterio 3 — pendiente

Igual que el criterio 2 de la marca de agua: necesita dos fotos del dataset en
días distintos para comprobar que un mismo contrato conserva su `id_contrato`.

### Conclusión de la sección 3

Datos del 2026-10-09. Ninguna de las tres candidatas tiene nulos, así que el
criterio 1 no las distingue. El que decide es el criterio 2:

* `referencia_del_contrato`: ≈1,56 millones de repetidos. Es una referencia que
  pone cada entidad, y distintas entidades pueden repetir la misma. **Descartada.**
* `proceso_de_compra`: ≈712.000 repetidos. Es esperable: un proceso de compra
  puede adjudicar varios contratos. Identifica el proceso, no el contrato.
  **Descartada.**
* `id_contrato`: 306 repetidos, todos exactamente dos veces, y las 306 parejas son
  copias exactas. Deduplicar por copia exacta no pierde información.
  **Elegida**, con el criterio 3 (estabilidad) pendiente de verificar.

Si en una carga futura aparece un `id_contrato` repetido con filas distintas, la
deduplicación necesitará un criterio de versión, y ahí se acopla con la
sección 2.

## 4. Porcentaje de nulos por columna

Conteo exacto en el servidor: `count(columna)` cuenta solo los no nulos.
Se consulta en lotes para no armar una URL gigante.

In [14]:
def count_non_null(column_names):
    select = ", ".join(f"count({c}) as c{i}" for i, c in enumerate(column_names))
    result = soql({"$select": select}).iloc[0].astype(int)
    result.index = column_names
    return result


BATCH_SIZE = 15
field_names = columns["fieldName"].tolist()
batches = [field_names[i:i + BATCH_SIZE] for i in range(0, len(field_names), BATCH_SIZE)]
non_null_counts = pd.concat([count_non_null(batch) for batch in batches])

nulls = pd.DataFrame({
    "tipo": columns.set_index("fieldName")["dataTypeName"],
    "pct_nulos": ((1 - non_null_counts / total_rows) * 100).round(2),
}).sort_values("pct_nulos", ascending=False)
nulls[nulls["pct_nulos"] > 0]

,tipo,pct_nulos
fecha_fin_reversi_n,calendar_date,99.99
fecha_inicio_reversi_n,calendar_date,99.99
fecha_fin_obligaciones_posconsumo,calendar_date,99.97
fecha_inicio_obligaciones_posconsumo,calendar_date,99.97
fecha_de_notificaci_n_de_prorrogaci_n,calendar_date,90.36
fecha_inicio_liquidacion,calendar_date,89.03
fecha_fin_liquidacion,calendar_date,89.03
ultima_actualizacion,calendar_date,41.71
fecha_de_inicio_del_contrato,calendar_date,7.66
fecha_de_firma,calendar_date,7.02


In [15]:
print(f"Columnas sin nulos: {(nulls['pct_nulos'] == 0).sum()}")
print(f"Columnas con más del 50 % de nulos: {(nulls['pct_nulos'] > 50).sum()}")
print(f"Columnas 100 % nulas: {(nulls['pct_nulos'] == 100).sum()}")

Columnas sin nulos: 84
Columnas con más del 50 % de nulos: 7
Columnas 100 % nulas: 0


## Muestra de trabajo (para las preguntas 5 a 8)

50.000 filas paginadas de a 10.000, ordenadas por `:id` para que la paginación
sea estable. **No es una muestra aleatoria**: es el primer tramo según el orden
interno de Socrata. Sirve para descubrir formatos, no para estimar proporciones
con precisión.

Ojo: en la respuesta JSON, Socrata **omite** los campos nulos de cada fila. Por
eso se reindexa con la lista de columnas de los metadatos.

In [16]:
PAGE_SIZE = 10_000
SAMPLE_SIZE = 50_000

pages = []
for offset in range(0, SAMPLE_SIZE, PAGE_SIZE):
    pages.append(soql({"$limit": PAGE_SIZE, "$offset": offset, "$order": ":id"}))

sample = pd.concat(pages, ignore_index=True)
missing_in_json = set(field_names) - set(sample.columns)
sample = sample.reindex(columns=field_names)

print(sample.shape)
print(f"Columnas que no aparecieron en ninguna fila del JSON: {sorted(missing_in_json)}")

(50000, 95)
Columnas que no aparecieron en ninguna fila del JSON: []


In [17]:
# Columnas que no llegan como texto plano sino como objeto JSON anidado
nested_columns = [c for c in field_names if sample[c].map(lambda v: isinstance(v, dict)).any()]
print(f"Columnas con objetos anidados: {nested_columns}")
print(sample[nested_columns].dropna().head(2).to_dict("records"))

for col in nested_columns:
    sample[col] = sample[col].map(lambda v: json.dumps(v, ensure_ascii=False) if isinstance(v, dict) else v)

Columnas con objetos anidados: ['urlproceso']
[{'urlproceso': {'url': 'https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.1906043&isFromPublicArea=True&isModal=true&asPopupView=true'}}, {'urlproceso': {'url': 'https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.9909110&isFromPublicArea=True&isModal=true&asPopupView=true'}}]


## 5. Nulos disfrazados

Valores que significan "no hay dato" pero no son nulos. Primero se buscan los
sospechosos conocidos; después se mira el valor más frecuente de cada columna
para descubrir los que no estaban en la lista.

In [18]:
SUSPECT_VALUES = [
    "", "no definido", "sin descripcion", "n/a", "na", "no aplica", "ninguno", "ninguna",
    "null", "none", "sin informacion", "sin información", "-",
]

text_columns = columns.loc[columns["dataTypeName"] == "text", "fieldName"].tolist()

rows = []
for col in text_columns:
    normalized = sample[col].dropna().str.strip().str.lower()
    suspects = normalized[normalized.isin(SUSPECT_VALUES)]
    if len(suspects) > 0:
        rows.append({
            "columna": col,
            "nulos_reales": sample[col].isna().sum(),
            "nulos_disfrazados": len(suspects),
            "pct_disfrazados": round(len(suspects) / len(sample) * 100, 1),
            "valores": suspects.value_counts().to_dict(),
        })

disguised = pd.DataFrame(rows).sort_values("nulos_disfrazados", ascending=False)
disguised

,columna,nulos_reales,nulos_disfrazados,pct_disfrazados,valores
44,obligaciones_contractuales_o_poscontractuales_a_nivel_ambiental,0,49967,99.9,{'no definido': 49967}
45,uso_de_etiquetado_ecol_gico,0,49967,99.9,{'no definido': 49967}
42,requisitos_ambientales_en_las_especificaciones_t_cnicas,0,49967,99.9,{'no definido': 49967}
43,criterios_ambientales_en_la_evaluaci_n_de_las_ofertas,0,49967,99.9,{'no definido': 49967}
39,descripcion_documentos_tipo,0,49821,99.6,{'no definido': 49821}
16,puntos_del_acuerdo,0,49713,99.4,{'no aplica': 49713}
17,pilares_del_acuerdo,0,49713,99.4,{'no aplica': 49713}
41,criterios_de_sostenibilidad_ambiental,0,49244,98.5,{'no definido': 49244}
35,nombre_ordenador_de_pago,0,41886,83.8,{'no definido': 41886}
36,tipo_de_documento_ordenador_de_pago,0,41886,83.8,{'no definido': 41886}


In [19]:
most_common = []
for col in text_columns:
    counts = sample[col].value_counts()
    if len(counts) > 0:
        most_common.append({
            "columna": col,
            "valor_mas_comun": counts.index[0],
            "pct_muestra": round(counts.iloc[0] / len(sample) * 100, 1),
            "distintos": len(counts),
        })

pd.DataFrame(most_common).sort_values("pct_muestra", ascending=False).head(40)

,columna,valor_mas_comun,pct_muestra,distintos
27,reversion,No,100.0,2
26,obligaciones_postconsumo,No,100.0,2
62,uso_de_etiquetado_ecol_gico,No definido,99.9,3
59,requisitos_ambientales_en_las_especificaciones_t_cnicas,No definido,99.9,3
60,criterios_ambientales_en_la_evaluaci_n_de_las_ofertas,No definido,99.9,3
61,obligaciones_contractuales_o_poscontractuales_a_nivel_ambiental,No definido,99.9,3
56,descripcion_documentos_tipo,No definido,99.6,14
30,espostconflicto,No,99.4,2
32,pilares_del_acuerdo,No aplica,99.4,17
31,puntos_del_acuerdo,No aplica,99.4,6


## 6. Formatos de NIT y de valores monetarios

Técnica: convertir cada valor a su "forma" (dígito → `9`, letra → `A`) y contar
formas distintas. Así salen a la vista puntos, guiones y dígitos de verificación
sin revisar fila por fila.

In [20]:
def shape(value):
    if pd.isna(value):
        return "<nulo>"
    value = re.sub(r"[0-9]", "9", str(value))
    value = re.sub(r"[^\W\d_]+", "A", value)
    return value


sample["nit_entidad"].map(shape).value_counts().head(10)

nit_entidad
999999999     45702
9999999999     4283
99999999         15
Name: count, dtype: int64

In [21]:
# ¿Los NIT de 10 dígitos son NIT de 9 con el dígito de verificación pegado?
all_nits = soql({"$select": "nit_entidad", "$group": "nit_entidad", "$limit": 50000})["nit_entidad"]
nits_9 = set(all_nits[all_nits.str.len() == 9])
nits_10 = all_nits[all_nits.str.len() == 10]
with_check_digit = nits_10[nits_10.str[:9].isin(nits_9)]

print(f"NIT distintos: {len(all_nits)}")
print(f"De 9 dígitos: {len(nits_9)} | de 10 dígitos: {len(nits_10)}")
print(f"NIT de 10 dígitos cuyos primeros 9 también existen como NIT: {len(with_check_digit)}")

NIT distintos: 5767
De 9 dígitos: 5157 | de 10 dígitos: 602
NIT de 10 dígitos cuyos primeros 9 también existen como NIT: 130


In [22]:
provider_shapes = (
    sample.assign(forma=sample["documento_proveedor"].map(shape))
    .groupby("tipodocproveedor")["forma"]
    .value_counts()
    .groupby(level=0)
    .head(4)
)
provider_shapes

tipodocproveedor                 forma     
Cédula de Ciudadanía             9999999999    20801
                                 99999999      18051
                                 9999999        1706
                                 999999999        78
Cédula de Extranjería            999999           40
                                 9999999999       14
                                 9999999           8
                                 99999999          6
NIT                              999999999      6879
                                 9999999999      509
                                 99999999        185
                                 A A              53
No Definido                      A A            1394
Otro                             999999999        55
                                 A A              18
                                 9999999999       17
                                 99999999          5
Pasaporte                        999999999         2
  

In [23]:
money_columns = [
    c for c in columns.loc[columns["dataTypeName"] == "number", "fieldName"]
    if c.startswith(("valor_", "saldo_"))
]
money = sample[money_columns].apply(pd.to_numeric, errors="coerce")

pd.DataFrame({
    "no_convertibles": money.isna().sum() - sample[money_columns].isna().sum(),
    "negativos": (money < 0).sum(),
    "ceros": (money == 0).sum(),
    "con_decimales": (money % 1 != 0).sum(),
    "minimo": money.min(),
    "maximo": money.max(),
})

,no_convertibles,negativos,ceros,con_decimales,minimo,maximo
valor_del_contrato,0,0,1618,939,0.0,1.004004e+18
valor_de_pago_adelantado,0,0,49941,21,0.0,7.068767e+09
valor_facturado,0,0,23539,885,0.0,1.554781e+11
valor_pendiente_de_pago,0,25,15651,1158,-166502243.0,1.004004e+18
valor_pagado,0,0,26745,757,0.0,1.554781e+11
valor_amortizado,0,0,49986,8,0.0,9.306474e+08
valor_pendiente_de,0,1,49950,18,-1648487.7,7.068767e+09
valor_pendiente_de_ejecucion,0,25,15567,1158,-166502243.0,1.004004e+18
saldo_cdp,0,0,19429,0,0.0,2.546400e+15
saldo_vigencia,0,0,49170,0,0.0,4.095213e+12


In [24]:
sample["valor_del_contrato"].map(shape).value_counts().head(10)

valor_del_contrato
99999999        31953
9999999         11474
999999999        3038
9                1640
9999999999        672
99999999.99       352
999999999.99      157
999999            151
99999999.9        142
9999999.99        103
Name: count, dtype: int64

In [25]:
# Los valores más altos: ¿son contratos reales o errores de digitación?
sample.assign(valor=money["valor_del_contrato"]).nlargest(5, "valor")[
    ["id_contrato", "nombre_entidad", "tipo_de_contrato", "valor"]
]

,id_contrato,nombre_entidad,tipo_de_contrato,valor
30752,CO1.PCCNTR.6711801,HOSPITAL TIMBIO ESE,Decreto 092 de 2017,1.004004e+18
31390,CO1.PCCNTR.2940109,ALCALDIA MUNICIPIO DE IBAGUE*,Arrendamiento de inmuebles,1.120211e+16
15083,CO1.PCCNTR.1264602,MINISTERIO DE COMERCIO INDUSTRIA Y TURISMO - MINCIT,Prestación de servicios,1.060076e+16
26714,CO1.PCCNTR.3400242,ALCALDIA LOCAL DE KENNEDY,Prestación de servicios,6.203138e+15
46931,CO1.PCCNTR.10019404,INSTITUTO COLOMBIANO DE BIENESTAR FAMILIAR REGIONAL BOYACA,Otro,5.646850e+15


## 7. Formatos y zona horaria de las fechas

En Socrata, el tipo `calendar_date` es un *floating timestamp*: fecha y hora
**sin zona horaria**. La zona no se puede leer del dato; hay que asumirla
(hora de Colombia, UTC−5) y documentar el supuesto.

In [26]:
date_columns = columns.loc[columns["dataTypeName"] == "calendar_date", "fieldName"].tolist()

pd.DataFrame({
    col: sample[col].dropna().map(shape).value_counts().head(3).to_dict()
    for col in date_columns
}.items(), columns=["columna", "formas"])

,columna,formas
0,fecha_de_firma,{'9999-99-99A99:99:99.999': 46446}
1,fecha_de_inicio_del_contrato,{'9999-99-99A99:99:99.999': 46139}
2,fecha_de_fin_del_contrato,{'9999-99-99A99:99:99.999': 49520}
3,ultima_actualizacion,{'9999-99-99A99:99:99.999': 29069}
4,fecha_inicio_liquidacion,{'9999-99-99A99:99:99.999': 5451}
5,fecha_fin_liquidacion,{'9999-99-99A99:99:99.999': 5451}
6,fecha_de_notificaci_n_de_prorrogaci_n,{'9999-99-99A99:99:99.999': 4816}
7,fecha_inicio_obligaciones_posconsumo,{'9999-99-99A99:99:99.999': 11}
8,fecha_fin_obligaciones_posconsumo,{'9999-99-99A99:99:99.999': 11}
9,fecha_inicio_reversi_n,{'9999-99-99A99:99:99.999': 2}


In [27]:
dates = sample[date_columns].apply(pd.to_datetime, errors="coerce", format="ISO8601")

pd.DataFrame({
    "no_convertibles": dates.isna().sum() - sample[date_columns].isna().sum(),
    "con_hora": (dates.apply(lambda s: s.dt.time) != pd.Timestamp("00:00").time()).sum() - dates.isna().sum(),
    "minimo": dates.min(),
    "maximo": dates.max(),
})

,no_convertibles,con_hora,minimo,maximo
fecha_de_firma,0,0,2016-05-12 00:00:00,2026-10-06 00:00:00
fecha_de_inicio_del_contrato,0,0,2016-01-26 00:00:00,2026-12-23 00:00:00
fecha_de_fin_del_contrato,0,0,2014-01-24 00:00:00,2050-11-12 00:00:00
ultima_actualizacion,0,0,2017-08-24 00:00:00,2026-10-06 00:00:00
fecha_inicio_liquidacion,0,0,2012-07-22 00:00:00,2046-01-23 00:00:00
fecha_fin_liquidacion,0,0,2012-07-22 00:00:00,2047-11-30 00:00:00
fecha_de_notificaci_n_de_prorrogaci_n,0,0,2016-06-05 00:00:00,2038-06-27 00:00:00
fecha_inicio_obligaciones_posconsumo,0,10,2018-11-17 05:00:00,2025-07-01 05:00:00
fecha_fin_obligaciones_posconsumo,0,11,2019-05-17 05:00:00,2026-07-01 04:59:00
fecha_inicio_reversi_n,0,2,2024-08-30 17:00:00,2046-01-23 17:00:00


In [28]:
today = pd.Timestamp.today().normalize()

print("Fin antes del inicio:",
      (dates["fecha_de_fin_del_contrato"] < dates["fecha_de_inicio_del_contrato"]).sum())
print("Firma en el futuro:", (dates["fecha_de_firma"] > today).sum())
print("Firma antes de 2015 (SECOP II empezó a operar hacia 2015, no verificado):",
      (dates["fecha_de_firma"] < "2015-01-01").sum())

Fin antes del inicio: 4
Firma en el futuro: 0
Firma antes de 2015 (SECOP II empezó a operar hacia 2015, no verificado): 0


## 8. Normalización de nombres de entidades

Si `nit_entidad` identifica a la entidad, debería tener un solo nombre.

In [29]:
entities = soql({"$select": "count(distinct nit_entidad) as nits, count(distinct nombre_entidad) as nombres"})
entities

,nits,nombres
0,5767,6480


In [30]:
names_per_nit = soql({
    "$select": "nit_entidad, count(distinct nombre_entidad) as nombres",
    "$group": "nit_entidad",
    "$having": "count(distinct nombre_entidad) > 1",
    "$order": "nombres DESC",
    "$limit": 50000,
})
print(f"NIT con más de un nombre: {len(names_per_nit)}")
names_per_nit.head(10)

NIT con más de un nombre: 342


,nit_entidad,nombres
0,899999034,83
1,899999061,34
2,899999239,32
3,899999067,32
4,899999114,29
5,899999069,29
6,890399011,28
7,891900272,17
8,890399029,16
9,8903990295,12


In [31]:
top_nit = names_per_nit.loc[0, "nit_entidad"]
soql({
    "$select": "nombre_entidad, count(*) as filas",
    "$where": f"nit_entidad = {top_nit}",
    "$group": "nombre_entidad",
    "$order": "filas DESC",
    "$limit": 15,
})

,nombre_entidad,filas
0,SENA REGIONAL VALLE Grupo de Apoyo Administrativo Mixto,12400
1,SENA REGIONAL ANTIOQUIA Grupo de Apoyo Administrativo Mixto,11156
2,SENA SECRETARIA GENERAL,10740
3,SENA REGIONAL SANTANDER Grupo de Apoyo Administrativo Mixto,10001
4,SENA REGIONAL NORTE DE SANTANDER Grupo de Apoyo Administrativo Mixto,9357
5,SENA REGIONAL ATLANTICO Intercentros,9246
6,SENA REGIONAL TOLIMA 1,9017
7,SENA REGIONAL DISTRITO CAPITAL Grupo de Apoyo Administrativo Mixto CSF,8783
8,SENA REGIONAL DISTRITO CAPITAL Grupo Administrativo Intercentros CEET,8700
9,SENA REGIONAL CESAR Grupo de Apoyo Administrativo Mixto,8625


In [32]:
def normalize_name(name):
    name = unicodedata.normalize("NFKD", str(name))
    name = "".join(ch for ch in name if not unicodedata.combining(ch))
    name = re.sub(r"[^A-Za-z0-9 ]", " ", name.upper())
    return re.sub(r"\s+", " ", name).strip()


raw_names = sample["nombre_entidad"].nunique()
normalized_names = sample["nombre_entidad"].map(normalize_name).nunique()
print(f"Nombres distintos en la muestra: {raw_names}")
print(f"Después de normalizar mayúsculas, tildes y puntuación: {normalized_names}")

Nombres distintos en la muestra: 2625
Después de normalizar mayúsculas, tildes y puntuación: 2624


## Muestra de 100 filas para `data/samples/`

Se excluyen las columnas con datos personales (nombres y documentos de personas
naturales, cuentas bancarias). El dataset es público, pero republicar esos datos
en un repositorio no es necesario para el propósito de la muestra.

In [33]:
PERSONAL_DATA_COLUMNS = [
    "documento_proveedor", "proveedor_adjudicado", "codigo_proveedor",
    "nombre_representante_legal", "identificaci_n_representante_legal",
    "domicilio_representante_legal",
    "nombre_del_banco", "tipo_de_cuenta", "n_mero_de_cuenta",
    "nombre_ordenador_del_gasto", "n_mero_de_documento_ordenador_del_gasto",
    "nombre_supervisor", "n_mero_de_documento_supervisor",
    "nombre_ordenador_de_pago", "n_mero_de_documento_ordenador_de_pago",
]

sample_100 = sample.drop(columns=PERSONAL_DATA_COLUMNS).head(100)
output_path = ROOT / "data" / "samples" / "contracts_sample_100.csv"
sample_100.to_csv(output_path, index=False, encoding="utf-8")
print(f"{sample_100.shape} -> {output_path.relative_to(ROOT)}")

(100, 80) -> data\samples\contracts_sample_100.csv
